# Silver Layer - Netflix Titles Transformation

**Purpose:** Clean, enrich, validate, and write the main `netflix_titles` dataset to the Silver Delta layer.

This notebook reads the Bronze Delta version of `netflix_titles`, applies cleansing and enrichment rules, validates the transformed dataset, and writes the curated result to the Silver layer.

## 1. Environment Setup

Define the source and target paths used by the Silver transformation job.

In [ ]:
from pyspark.sql.functions import col, dense_rank, split, sum as spark_sum, to_date, trim, when
from pyspark.sql.types import IntegerType
from pyspark.sql.window import Window

BRONZE_TITLES_PATH = "abfss://bronze@nextflixprojectdltluc.dfs.core.windows.net/netflix_titles"
SILVER_TITLES_PATH = "abfss://silver@nextflixprojectdltluc.dfs.core.windows.net/netflix_titles"

## 2. Read Bronze Data

Load the raw Delta dataset produced by Auto Loader.

In [ ]:
bronze_df = spark.read.format("delta").load(BRONZE_TITLES_PATH)

display(bronze_df.limit(10))
bronze_df.printSchema()

## 3. Clean and Enrich Titles

Standardize strings, convert numeric/date fields, fill expected nulls, create reporting-friendly columns, and rank titles by movie duration.

In [ ]:
silver_df = (
    bronze_df
    .withColumn("show_id", trim(col("show_id")))
    .withColumn("type", trim(col("type")))
    .withColumn("title", trim(col("title")))
    .withColumn("rating", trim(col("rating")))
    .withColumn("short_title", trim(split(col("title"), ":").getItem(0)))
    .withColumn("rating_group", split(col("rating"), "-").getItem(0))
    .withColumn("date_added", to_date(col("date_added"), "M/d/yyyy"))
    .withColumn("release_year", col("release_year").cast(IntegerType()))
    .withColumn("duration_minutes", col("duration_minutes").cast(IntegerType()))
    .withColumn("duration_seasons", col("duration_seasons").cast(IntegerType()))
    .fillna({"duration_minutes": 0, "duration_seasons": 0, "rating": "Unknown"})
    .withColumn(
        "type_flag",
        when(col("type") == "Movie", 1)
        .when(col("type") == "TV Show", 2)
        .otherwise(0)
    )
)

duration_window = Window.orderBy(col("duration_minutes").desc())
silver_df = silver_df.withColumn("duration_ranking", dense_rank().over(duration_window))

display(silver_df.limit(10))

## 4. Data Quality Checks

Fail fast if the transformed dataset is empty, has duplicate title IDs, or is missing required values for downstream Gold aggregations.

In [ ]:
required_columns = ["show_id", "type", "title", "release_year"]

row_count = silver_df.count()
duplicate_show_ids = silver_df.groupBy("show_id").count().filter(col("count") > 1).count()
null_check = silver_df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in required_columns
]).collect()[0].asDict()

assert row_count > 0, "Silver titles dataset is empty."
assert duplicate_show_ids == 0, f"Found {duplicate_show_ids} duplicate show_id values."
assert all(v == 0 for v in null_check.values()), f"Required columns contain nulls: {null_check}"

quality_summary = spark.createDataFrame([
    (row_count, duplicate_show_ids, str(null_check))
], ["row_count", "duplicate_show_ids", "required_column_nulls"])

display(quality_summary)

## 5. Write Silver Delta Table

Overwrite the curated Silver dataset so Gold notebooks and Power BI-facing aggregations always read the latest trusted version.

In [ ]:
try:
    silver_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .option("path", SILVER_TITLES_PATH) \
        .save()
    
    print(f"Silver titles written to {SILVER_TITLES_PATH}")
except Exception as e:
    print(f'Error: {str(e)}')
    raise e


## 6. Optional Delta Merge Pattern

For a production incremental pipeline, replace full overwrite with a Delta `MERGE` keyed by `show_id`. Keep this pattern ready for interview discussion and future enhancement.

In [ ]:
# from delta.tables import DeltaTable
#
# target = DeltaTable.forPath(spark, SILVER_TITLES_PATH)
# (
#     target.alias("t")
#     .merge(silver_df.alias("s"), "t.show_id = s.show_id")
#     .whenMatchedUpdateAll()
#     .whenNotMatchedInsertAll()
#     .execute()
# )